# Chicago Closure Radar — Step 3: Feature Engineering

Build the full feature matrix for model training.
Uses a **snapshot-based** approach: pick a historical date, compute features up to that date,
then label whether the business closed within the next 6 months.

This simulates real-world deployment: "if we ran the model today, what would it predict?"

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

biz = pd.read_parquet('../data/processed/yelp_chicago_businesses.parquet')
rev = pd.read_parquet('../data/processed/yelp_chicago_reviews.parquet')
gt  = pd.read_parquet('../data/ground_truth/ground_truth.parquet')

print(f'Businesses: {len(biz):,} | Reviews: {len(rev):,}')
print(f'Review date range: {rev["date"].min().date()} → {rev["date"].max().date()}')

## Build Feature Matrix at Snapshot Date

In [ ]:
from src.features.build_features import build_feature_matrix

SNAPSHOT = pd.Timestamp('2022-01-01')
HORIZON  = 180  # predict closure within 6 months

matrix = build_feature_matrix(
    businesses=biz,
    reviews=rev,
    ground_truth=gt,
    snapshot_date=SNAPSHOT,
    horizon_days=HORIZON,
    sentiment_model='vader',
)

out_path = f'../data/processed/features_{SNAPSHOT.date()}_{HORIZON}d.parquet'
matrix.to_parquet(out_path, index=False)
print(f'Saved to {out_path}')
print(f'Label balance: {matrix["label"].value_counts().to_dict()}')
matrix.head(3)

## Feature Correlation Heatmap

In [ ]:
feature_cols = [c for c in matrix.columns if c not in ['business_id', 'name', 'categories',
                                                          'label', 'closure_date', 'latitude',
                                                          'longitude', 'is_open']]
corr = matrix[feature_cols + ['label']].corr()['label'].drop('label').sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
corr.plot.barh(ax=ax, color=['tomato' if v > 0 else 'steelblue' for v in corr.values])
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Feature Correlation with Closure Label')
ax.set_xlabel('Pearson r with label')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_correlation.png', dpi=150)
plt.show()

## Multiple Snapshots (for Robust Training)

In [ ]:
# Build features at multiple dates to get more labeled examples
snapshots = ['2020-01-01', '2021-01-01', '2022-01-01']
all_matrices = []

for snap in snapshots:
    print(f'Building snapshot: {snap}')
    m = build_feature_matrix(biz, rev, gt, pd.Timestamp(snap), 180, 'vader')
    m['snapshot_date'] = snap
    all_matrices.append(m)

combined = pd.concat(all_matrices, ignore_index=True)
combined.to_parquet('../data/processed/features_combined.parquet', index=False)
print(f'Combined matrix: {len(combined):,} rows | Positive labels: {combined["label"].sum():,}')